# 3DD-TTA Colab Kurulum ve Test Rehberi
Bu not defteri, 3DD-TTA projesini Google Colab üzerinde çalıştırmak için özel olarak hazırlanmıştır.
Tüm bağımlılıklar ve eklentiler (EMD, Chamfer, PointNet++ Ops) Colab'daki herhangi bir GPU (T4, L4, A100) ile uyumlu olacak şekilde derlenir.
Gerekli tüm kod düzeltmeleri (sed işlemleri) doğrudan GitHub reposundaki `dev` branşında yapıldığı için kurulum tamamen sadeleştirilmiştir.

### Adım 1: Conda Ortamını Hazırlama
Colab üzerinde Python 3.8 ortamını oluşturmak için `condacolab` kuruyoruz. Bu hücre çalıştıktan sonra kernel otomatik olarak yeniden başlayabilir.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

### Adım 2: Repoyu Klonlama ve `3dd_tta_env` Ortamını Oluşturma
Projeyi klonluyor, güncel düzeltmeleri içeren `dev` branşına geçiyor ve `env.yaml` dosyasından yazarın orijinal ortamını kuruyoruz.

In [ ]:
import condacolab
condacolab.check()

# Projeyi klonla ve dev branşına geç
!git clone https://github.com/BatuhanOrhon/3DD-TTA.git
%cd 3DD-TTA
!git checkout dev

# 1. Yazarın 3dd_tta_env (Python 3.8) ortamını env.yaml dosyasından yaratıyoruz:
!conda env create -f env.yaml

### Google Drive Bağlantısı ve KNN_CUDA Dosyasının Alınması
Google Drive a bağlanıp `thesis` klasöründeki `.whl` dosyasını Colab ortamına kopyalıyoruz.

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source_path = "/content/drive/MyDrive/thesis/KNN_CUDA-0.2-py3-none-any.whl"
dest_path = "/content/KNN_CUDA-0.2-py3-none-any.whl"

if os.path.exists(source_path):
    shutil.copy(source_path, dest_path)
    print(f"✅ Dosya başarıyla kopyalandı: {dest_path}")
else:
    print(f"❌ HATA: Kaynak dosya bulunamadı: {source_path}")

### Adım 3: Kütüphanelerin Kurulumu ve C++/CUDA Eklentilerinin Derlenmesi
Tüm gerekli kütüphaneleri kuruyor ve eklentileri derliyoruz.

In [ ]:
# 1. Gerekli kütüphaneleri izole ortama kur
!conda run -n 3dd_tta_env pip install easydict open3d pyyaml tensorboardX timm==0.4.5 tqdm transforms3d termcolor wandb loguru einops comet_ml calmsize diffusers tabulate ninja h5py
# 2. KNN_CUDA kurulumu (/content altında bulunan .whl dosyası)
!conda run -n 3dd_tta_env pip install /content/KNN_CUDA-0.2-py3-none-any.whl
# 3. requirements.txt dosyasındaki diğer bağımlılıklar
!conda run -n 3dd_tta_env pip install -r requirements.txt
# 4. EMD Extension derlemesi
%cd extensions/emd
!TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6" conda run -n 3dd_tta_env python setup.py install
%cd ../..
# 5. Chamfer Distance derlemesi
%cd extensions/chamfer_dist
!conda run -n 3dd_tta_env python setup.py install
%cd ../..
# 6. PointNet++ Ops derlemesi (Colab GPU'ları ile tam uyumlu derleme)
%cd Pointnet2_PyTorch/pointnet2_ops_lib
!TORCH_CUDA_ARCH_LIST="7.0;7.5;8.0;8.6" conda run -n 3dd_tta_env python setup.py install
%cd ../..
# 7. CLIP kurulumu
!conda run -n 3dd_tta_env pip install git+https://github.com/openai/CLIP.git
# 8. Proje paketini derle
!conda run -n 3dd_tta_env python build_pkg.py
print("✅ Tüm derleme ve kurulumlar kesin olarak Python 3.8 ortamında yapıldı!")


### Adım 3.5: ModelNet40 ve ModelNet40-C Veri Yönetimi
Öncelikle verilerin Drive`da bulunup bulunmadığını kontrol ediyoruz.
- Eğer veriler `/content/drive/MyDrive/thesis/dataset` dizininde varsa, tekrar indirmek yerine direkt Colab`a kopyalıyoruz.
- Eğer yoksa, indiriyor ve bir sonraki sefer için Drive`a kopyalayarak yedekliyoruz.

In [ ]:
import os
import shutil

os.makedirs("./data", exist_ok=True)
drive_dataset_dir = "/content/drive/MyDrive/thesis/dataset"
os.makedirs(drive_dataset_dir, exist_ok=True)

# Sadece Corrupted Veriseti Icin (Orijinal veriseti TTA icin gereksizdir!)
corrupted_zip_drive = os.path.join(drive_dataset_dir, "modelnet40_c.zip")
corrupted_zip_local = "./data/modelnet40_c.zip"

if os.path.exists(corrupted_zip_drive):
    print("ModelNet40-C Corrupted verisi Drive`dan kopyalanıyor...")
    shutil.copy(corrupted_zip_drive, corrupted_zip_local)
else:
    print("ModelNet40-C Corrupted verisi indiriliyor...")
    !wget -O {corrupted_zip_local} "https://zenodo.org/records/6017834/files/modelnet40_c.zip"
    print("Drive`a yedekleniyor...")
    shutil.copy(corrupted_zip_local, corrupted_zip_drive)

!unzip -q -o {corrupted_zip_local} -d ./data/
print("Veri setleri başarıyla Colab ortamına hazırlandı.")


### Adım 4: Model Ağırlıklarının İndirilmesi
LION ve PointMAE ağırlıklarını indiriyoruz. `gdown`'ın oluşturabileceği alt klasör yapısını otomatik düzeltiyoruz.

In [ ]:
import os
import shutil
!pip install -q gdown
local_pointnet_dir = "/content/3DD-TTA/pointnet_ckpts"
local_lion_dir = "/content/3DD-TTA/lion_ckpts"
os.makedirs(local_pointnet_dir, exist_ok=True)
os.makedirs(local_lion_dir, exist_ok=True)
drive_weights_dir = "/content/drive/MyDrive/thesis/weights"
os.makedirs(drive_weights_dir, exist_ok=True)
# 1. PointMAE Weights
pointmae_drive = os.path.join(drive_weights_dir, "modelnet_jt.pth")
pointmae_local = os.path.join(local_pointnet_dir, "modelnet_jt.pth")
if os.path.exists(pointmae_drive):
    print("PointMAE ağırlıkları Drive'dan kopyalanıyor...")
    shutil.copy(pointmae_drive, pointmae_local)
else:
    print("PointMAE ağırlıkları indiriliyor...")
    !gdown --folder https://drive.google.com/drive/folders/1MTH8WpOqfAIiZ0DZV9p-tSKiDgQ_Id5A?usp=sharing -O /content/temp_pointnet/
    !find /content/temp_pointnet -name "modelnet_jt.pth" -exec mv {} {local_pointnet_dir}/ \;
    !rm -rf /content/temp_pointnet
    print("PointMAE ağırlıkları Drive'a yedekleniyor...")
    if os.path.exists(pointmae_local):
        shutil.copy(pointmae_local, pointmae_drive)
# 2. LION Diffusion Weights
lion_drive = os.path.join(drive_weights_dir, "epoch_10999_iters_2100999.pt")
lion_local = os.path.join(local_lion_dir, "epoch_10999_iters_2100999.pt")
if os.path.exists(lion_drive):
    print("LION ağırlıkları Drive'dan kopyalanıyor...")
    shutil.copy(lion_drive, lion_local)
else:
    print("LION ağırlıkları indiriliyor...")
    !wget -O {lion_local} -nc https://huggingface.co/xiaohui2022/lion_ckpt/resolve/main/unconditional/all55/checkpoints/epoch_10999_iters_2100999.pt
    print("LION ağırlıkları Drive'a yedekleniyor...")
    if os.path.exists(lion_local):
        shutil.copy(lion_local, lion_drive)
print("✅ Model ağırlıkları (LION ve PointMAE) kullanıma hazır!")


### Adım 5: Çıkarım (Inference) Scriptinin Oluşturulması
`test_inference.py` dosyasını oluşturuyoruz. Tüm GPU'larla uyumlu ve TTA akışını baştan sona test edecek mimaridedir.

In [ ]:
%%writefile test_inference.py
import torch
import numpy as np
import sys
import os
from default_config import cfg as diff_config
from utils_mate.config import cfg_from_yaml_file
from default_config import cfg as configs
from models.lion import LION
from utilities_3dd_tta import load_base_model, normalize, rotate_pointcloud, rotateback_pointcloud
from tta import tta_reconstruct
from graph_spectral import GraphSpectralDNA
from utils_mate import misc

print(f"Çalışan Python Sürümü: {sys.version}")
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.0;7.5;8.0;8.6"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Modeller yükleniyor...")
pm_cfg = cfg_from_yaml_file("./cfgs/tta_modelnet.yaml")
pm_cfg.model.cls_dim = 40
class Args: pass
args = Args()
args.pointmae_ckpt = "./pointnet_ckpts/modelnet_jt.pth"
args.use_gpu = torch.cuda.is_available()
args.distributed = False
base_model = load_base_model(args, pm_cfg, None)
base_model.eval()

diff_config.merge_from_file("./lion_ckpts/unconditional_all55_cfg.yml")
diff_model = LION(configs)
diff_model.load_model("./lion_ckpts/epoch_10999_iters_2100999.pt")
graph_spectral_module = GraphSpectralDNA(k=10, delta=0.1, gamma=0.6, M=100, use_4d_gft=False, device=device)
print("Modeller başarıyla yüklendi!")

dummy_data = torch.rand(1, 2048, 3)
print(f"Girdi nokta bulutu boyutu: {dummy_data.shape}")

data_sample, data_center, data_max = normalize(dummy_data)
data_sample = data_sample.float().to(device)
data_sample *= 3.3885
data_sample = rotate_pointcloud(data_sample)

print("TTA (Test-Time Adaptation) işlemi başlıyor...")
gamma, eta, lambdaa = 0.01, 0.01, 0.95
num_steps = 5
pred_points = tta_reconstruct(data_sample, diff_model, graph_spectral_module, num_steps, gamma, eta, lambdaa, {"spectral": 1.0, "chamfer": 0.0}, total=100)
print(f"TTA tamamlandı. Çıktı boyutu: {pred_points.shape}")

pred_points = rotateback_pointcloud(pred_points)
pred_points, _, _ = normalize(pred_points)
pred_points = misc.fps(pred_points, 1024)

print("Sınıflandırma yapılıyor...")
with torch.no_grad():
    logits = base_model.module.classification_only(pred_points, only_unmasked=False)
    pred_class = logits.argmax(-1).item()

print(f"✅ Başarılı! Modelin bu rastgele nokta bulutu için tahmin ettiği sınıf ID'si: {pred_class}")

### Adım 6: Testi Çalıştırma
Oluşturduğumuz scripti `3dd_tta_env` ortamında çalıştırıyoruz.

In [ ]:
!conda run -n 3dd_tta_env python test_inference.py


### Adım 6: Qualitative Evaluation (Görsel Değerlendirme)
`README.md` dosyasında bahsedilen kalitatif (görsel) değerlendirme adımını çalıştırıyoruz.
Bu komut, belirli bir örneklem (`sample_id`) üzerinde TTA işlemini yapar ve sonuçları `./outputs/qualitative` klasörüne kaydeder.

In [ ]:
!conda run --no-capture-output -n 3dd_tta_env python demo_gsd_tta.py \
  --diff_ckpt ./lion_ckpts/epoch_10999_iters_2100999.pt \
  --denoising_step 35 \
  --dataset_root ./data/modelnet40_c \
  --corruption background \
  --sample_id 11


### Adım 7: GSDTTA Quantitative Test on ModelNet40-C
Yeni entegre edilen Graph Spectral (GSDTTA) algoritmasını ModelNet40-C veri setindeki çeşitli bozulmalar (corruptions) üzerinde çalıştırıyoruz.
Aşağıdaki komut `--use_4d_gft` bayrağını ekleyerek veya çıkararak 3D ve 4D spektral testler yapmanıza olanak tanır.

In [ ]:
!conda run --no-capture-output -n 3dd_tta_env python main_gsd_tta.py \
  --dataset_name modelnet-c \
  --dataset_root ./data/modelnet40_c \
  --label_path ./data/modelnet40_c/label.npy \
  --pointmae_config ./cfgs/tta_modelnet.yaml \
  --pointmae_ckpt ./pointnet_ckpts/modelnet_jt.pth \
  --batch_size 16 \
  --weight_spectral 1.0 \
  --weight_chamfer 0.0
  # 4D GFT test etmek icin komutun sonuna --use_4d_gft ekleyebilirsiniz


### Adım 8: Sonuçları Google Drive'a Yedekleme
Hem nicel (sayısal) analiz sonuçlarını (`gsd_results.txt`) hem de nitel (görsel) GIF çıktılarını kalıcı olarak Drive hesabınızdaki `thesis/outputs` klasörüne aktarıyoruz.

In [ ]:
import os
import shutil

# Drive'da hedeflenen klasörleri oluştur
drive_outputs_dir = "/content/drive/MyDrive/thesis/outputs"
drive_quant_dir = os.path.join(drive_outputs_dir, "quantitative")
drive_qual_dir = os.path.join(drive_outputs_dir, "qualitative")
os.makedirs(drive_quant_dir, exist_ok=True)
os.makedirs(drive_qual_dir, exist_ok=True)

# 1. Quantitative (Nicel) logları yedekle
local_results_file = "./outputs/quantitative/gsd_results.txt"
if os.path.exists(local_results_file):
    shutil.copy(local_results_file, os.path.join(drive_quant_dir, "gsd_results.txt"))
    print("✅ Sayısal sonuçlar (gsd_results.txt) Drive'a aktarıldı!")
else:
    print("⚠️ gsd_results.txt henüz oluşturulmamış.")

# 2. Qualitative (Görsel/GIF) çıktıları yedekle
local_qual_dir = "./outputs/qualitative"
if os.path.exists(local_qual_dir):
    files = os.listdir(local_qual_dir)
    gif_count = 0
    for f in files:
        if f.endswith(".gif"):
            shutil.copy(os.path.join(local_qual_dir, f), os.path.join(drive_qual_dir, f))
            gif_count += 1
    if gif_count > 0:
        print(f"✅ {gif_count} adet GIF dosyası Drive'a başarıyla aktarıldı!")
    else:
        print("ℹ️ Qualitative klasöründe GIF dosyası bulunamadı.")
else:
    print("⚠️ Qualitative klasörü henüz oluşturulmamış.")

### Adım 9: Orijinal 3DD-TTA (Baseline) Karşılaştırma Testleri
Bu adımda yazarların makalede kullandığı orijinal Chamfer Distance güdümlü (GSDTTA spektral loss olmadan) TTA algoritmasını çalıştırıyoruz.
Böylece kendi önerdiğimiz Graph Spectral (GSDTTA) yöntemiyle orijinal yöntemin sonuçlarını (hem görsel hem sayısal olarak) kıyaslayabileceksiniz.

In [ ]:
# 1. Baseline Qualitative (Görsel) Testi
# Sadece Chamfer Distance weight kullanıyoruz, Spectral weight = 0.0
!conda run --no-capture-output -n 3dd_tta_env python demo_3dd_tta.py \
  --diff_ckpt ./lion_ckpts/epoch_10999_iters_2100999.pt \
  --denoising_step 35 \
  --dataset_root ./data/modelnet40_c \
  --corruption background \
  --sample_id 11


In [ ]:
# 2. Baseline Quantitative (Sayısal) Testi on ModelNet40-C
# Yine aynı şekilde Spectral = 0.0, Chamfer = 1.0 ile accuracy ölçümü yapıyoruz.
!conda run --no-capture-output -n 3dd_tta_env python main_3dd_tta.py \
  --dataset_name modelnet-c \
  --dataset_root ./data/modelnet40_c \
  --label_path ./data/modelnet40_c/label.npy \
  --pointmae_config ./cfgs/tta_modelnet.yaml \
  --pointmae_ckpt ./pointnet_ckpts/modelnet_jt.pth \
  --batch_size 16


### Adım 10: Graph Spectral Loss Analizi ve Görselleştirme (Parametrik)
Bu hücre, `GraphSpectralAnalyzer` sınıfını kullanarak TTA döngüsü boyunca $H$ matrislerinin (düşük frekanslar) ve nokta bulutunun (spatial) hedefe ne kadar yaklaştığını (Chamfer ve MSE cinsinden) analiz eder ve grafik çizdirir. Tüm GFT ve difüzyon parametrelerini buradan ayarlayabilirsiniz.

In [ ]:
# Bu hücre conda ortamında demo_analyzer.py scriptini çalıştırır ve üretilen grafiği ekrana basar.
!conda run --no-capture-output -n 3dd_tta_env python demo_analyzer.py \
  --diff_ckpt ./lion_ckpts/epoch_10999_iters_2100999.pt \
  --dataset_root ./data/modelnet40_c \
  --corruption background \
  --sample_id 11 \
  --denoising_step 30 \
  --weight_spectral 1.0 \
  --gamma_lr 1000 \
  --M 100 \
  --output_img analyzer_plot.png

from IPython.display import Image, display
display(Image("analyzer_plot.png"))


### Adım 11: Parametre Etki Analizi (Hyperparameter Sweep)
Bu hücre, belirli bir parametrenin (örneğin `gamma_lr`, `weight_spectral` veya `M`) farklı değerleri için `GraphSpectralAnalyzer` çalıştırır ve sonuçları (Spatial Chamfer Distance) tek bir grafik üzerinde karşılaştırmalı olarak çizer. Hangi parametrenin GSDTTA üzerinde en iyi etkiyi yarattığını bulmak için kullanılır.

In [ ]:
# Bu hücre conda ortamında demo_sweep.py scriptini çalıştırır ve üretilen grafiği ekrana basar.
!conda run --no-capture-output -n 3dd_tta_env python demo_sweep.py \
  --diff_ckpt ./lion_ckpts/epoch_10999_iters_2100999.pt \
  --dataset_root ./data/modelnet40_c \
  --corruption background \
  --sample_id 11 \
  --denoising_step 30 \
  --sweep_param weight_spectral \
  --sweep_values 0.1 1.0 10.0 50.0 \
  --output_img sweep_plot.png

from IPython.display import Image, display
display(Image("sweep_plot.png"))
